# Analyze Train Metrics

Set of visualization tools to check if run is healthy. Requires training with `--log-metrics`.

In [ ]:
%matplotlib widget

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
RUN_DIR = os.path.expanduser("~/.cache/nanorepro/runs/default")
assert os.path.exists(RUN_DIR)

In [ ]:
# List files
log_files = [fn for fn in os.listdir(RUN_DIR) if fn.startswith("train_log_") and fn.endswith(".jsonl")]
log_files = sorted(log_files)  # sort files to ensure rank order
print(log_files)

In [ ]:
log_objects = []
df_list = []

for log_file in log_files:
    with open(os.path.join(RUN_DIR, log_file), "r") as f:
        lines = f.readlines()
        metrics_all = []
        for line in lines:
            log_obj = json.loads(line)
            log_objects.append(log_obj)
            if log_obj['event'] == 'train':
                for m in log_obj['metrics']:
                    metrics_all.append({'step': log_obj['step'], 'rank': log_obj['rank'], **m})
                del log_obj['metrics']
        df_list.append(pd.DataFrame(metrics_all))
        print(f"{log_file}: {len(lines)} lines")
        del metrics_all

df = pd.concat(df_list, ignore_index=True)
df["block"] = df["block"].astype("Int64")
num_layers = df["block"].max() + 1
del df_list

In [ ]:
# Max step on each rank - needed when post-processing logs from active run
max_step_rank_0 = max([log_obj['step'] for log_obj in log_objects if log_obj['step'] is not None and log_obj['rank'] == 0])
log_objects = [log_obj for log_obj in log_objects if log_obj['step'] is None or log_obj['step'] <= max_step_rank_0]
df = df[df['step'] <= max_step_rank_0]

In [ ]:
# First log object on rank0 is user_config
# user_config params are raw as they were passed to the run via terminal and/or defaults from argparse, before any scaling or processing
run_config = log_objects[0]
print(json.dumps(run_config, indent=2))

In [ ]:
RUN_SUBTITLE = f"target_flops={run_config['target_flops']:.0e} depth={run_config['depth']}"
print(f"Run subtitle: {RUN_SUBTITLE}")

## Basic Metrics - BPB, CORE, loss, lr

In [ ]:
# -----------------
#     BPB Eval
# Prepare
log_bpb_eval = [obj for obj in log_objects if obj['event'] == 'bpb_eval']
if log_bpb_eval:
    df_bpb_eval = pd.DataFrame(log_bpb_eval)
    # Plot train loss by rank
    plt.figure(figsize=(12, 5))
    for rank, g in df_bpb_eval.groupby("rank"):  # should log on rank 0 only, but just in case
        plt.plot(g["step"][1:], g["val/bpb"][1:], label=f"rank {rank}")
    plt.title(f"BPB Eval by Rank ({RUN_SUBTITLE})")
    plt.xlabel("step")
    plt.ylabel("BPB")
    plt.legend(ncol=2, fontsize=8)
    plt.savefig("analyze_train_metrics_plot01_bpb_eval.png", dpi=200, bbox_inches="tight")
    plt.show()
    del log_bpb_eval, df_bpb_eval
else:
    print("No BPB eval logs found.")

In [ ]:
# -----------------
#    CORE Metric
log_core = [obj for obj in log_objects if obj['event'] == 'core_metric']
df_core = pd.DataFrame(log_core)

# Plot CORE by rank
if len(df_core) > 0:
    plt.figure(figsize=(12, 5))
    for rank, g in df_core.groupby("rank"):
        plt.plot(g["step"], g["core_metric"], marker='o', label=f"rank {rank}")
    plt.title(f"CORE Metric by Rank ({RUN_SUBTITLE})")
    plt.xlabel("step")
    plt.ylabel("CORE Metric")
    plt.legend(ncol=2, fontsize=8)
    plt.savefig("analyze_train_metrics_plot02_core_metric.png", dpi=200, bbox_inches="tight")
    plt.show()
    del log_core, df_core
else:
    print("No core metric logs found.")  # may be missing early in a run

In [ ]:
# Extract train logs w/o metrics for simplicity - train_loss, smooth_train_loss, lrm etc.
log_train_wo_metrics = [{k: v for k, v in obj.items() if k != "metrics"} for obj in log_objects if obj['event'] == 'train']
df_train_wo_metrics = pd.DataFrame(log_train_wo_metrics)
df_train_wo_metrics.head()

# --------------------------------------------
#    Total Train Loss (reduced across ranks)
plt.figure(figsize=(12, 5))
for rank, g in df_train_wo_metrics.groupby("rank"):
    plt.plot(g["step"], g["train/train_loss"], label=f"rank {rank}")
    break  # only need to plot one rank since they are already reduced across ranks
plt.title(f"Train Loss (reduced across ranks, {RUN_SUBTITLE})")
plt.xlabel("step")
plt.ylabel("RMS")
#plt.ylim(2.5, 3.5)
plt.legend(ncol=2, fontsize=8)
plt.savefig(f"analyze_train_metrics_plot03_train_loss.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------
#     Debiased Smooth Train Loss (per rank)
plt.figure(figsize=(12, 5))
for rank, g in df_train_wo_metrics.groupby("rank"):
    plt.plot(g["step"], g["train/smooth_train_loss"], label=f"rank {rank}")
plt.title(f"Debiased Smooth Train Loss by Rank ({RUN_SUBTITLE})")
plt.xlabel("step")
plt.ylabel("RMS")
#plt.ylim(2.5, 3.5)
plt.legend(ncol=2, fontsize=8)
plt.savefig(f"analyze_train_metrics_plot04_debiased_smooth_train_loss.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# -----------------------------
#   LR Multiplier
plt.figure(figsize=(12, 5))
for rank, g in df_train_wo_metrics.groupby("rank"):
    plt.plot(g["step"], g["train/lrm"], label=f"rank {rank}")
plt.title(f"LR Multiplier by Rank ({RUN_SUBTITLE})")
plt.xlabel("step")
plt.ylabel("LR Multiplier")
plt.legend(ncol=2, fontsize=8)
plt.savefig(f"analyze_train_metrics_plot05_lr_multiplier.png", dpi=200, bbox_inches="tight")
plt.show()
del log_train_wo_metrics, df_train_wo_metrics

## Detailed Train Metrics

In [ ]:
# ---------------------------------------------
#      Post-Block Residual RMS by Layer
# Prepare forward residuals for plotting
df_fwd_filtered = df[(df["tensor_name"] == "resid_post") & (df["surface"] == "fwd")].copy()  # filter fwd activations only
df_fwd_summed = df_fwd_filtered.groupby(["step", "block", "stat"], as_index=False)["value"].sum()  # sum over ranks
df_fwd_pivoted = df_fwd_summed.pivot(index=["step", "block"], columns="stat", values="value").reset_index()  # put sq_sum and num_el side by side
df_fwd_pivoted["rms"] = np.sqrt(df_fwd_pivoted["sq_sum"] / df_fwd_pivoted["num_el"])  # compute RMS
display(df_fwd_pivoted.head())
# Plot forward residual RMS by block
plt.figure(figsize=(12, 5))
for block_id, g in df_fwd_pivoted.groupby("block"):
    plt.plot(g["step"], g["rms"], label=f"layer {block_id}")
plt.title(f"Post-Block Residual RMS by Layer ({RUN_SUBTITLE})")
plt.xlabel("step")
plt.ylabel("RMS")
plt.legend(ncol=2, fontsize=8)
plt.savefig(f"analyze_train_metrics_plot06_post_block_residual_rms.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# --------------------------------------------
#    Param Gradient RMS by Layer (attn+mlp)
# Exclude "value_embed.weight", "attn.ve_gate.weight", not part of core block
core_block_tensors = ["attn.c_q.weight", "attn.c_k.weight", "attn.c_v.weight", "attn.c_proj.weight", "mlp.c_fc.weight", "mlp.c_proj.weight"]
# Prepare gradient norms for plotting
df_grad_filtered = df[(df['surface'] == 'grad') & (df["tensor_name"].isin(core_block_tensors))].copy()
df_grad_summed = df_grad_filtered.groupby(["step", "block", "stat"], as_index=False)["value"].sum()  # sum over ranks
df_grad_pivoted = df_grad_summed.pivot(index=["step", "block"], columns="stat", values="value").reset_index()  # put sq_sum and num_el side by side
df_grad_pivoted["rms"] = np.sqrt(df_grad_pivoted["sq_sum"] / df_grad_pivoted["num_el"])  # compute RMS
display(df_grad_pivoted.head())
# Plot gradient RMS by layer
plt.figure(figsize=(12, 5))
for block_id, g in df_grad_pivoted.groupby("block"):
    plt.plot(g["step"], g["rms"], label=f"layer {block_id}")
plt.title(f"Param Gradient RMS by Layer (attn+mlp {RUN_SUBTITLE})")
plt.xlabel("step")
plt.ylabel("RMS")
plt.yscale("log")
plt.legend(ncol=2, fontsize=8)
plt.savefig(f"analyze_train_metrics_plot07_param_gradient_rms.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# -------------------------------------------------
#     Parameter Update Ratio by Layer (attn+mlp)
# Exclude "value_embed.weight", "attn.ve_gate.weight", not part of core block
core_block_tensors = ["attn.c_q.weight", "attn.c_k.weight", "attn.c_v.weight", "attn.c_proj.weight", "mlp.c_fc.weight", "mlp.c_proj.weight"]
# Prepare update norms for plotting
df_update_filtered = df[(df['surface'].isin(['update', 'params'])) & (df['stat'] == 'sq_sum') & (df["tensor_name"].isin(core_block_tensors))].copy()
df_update_summed = df_update_filtered.groupby(["step", "block", "surface"], as_index=False)["value"].sum()  # sum over ranks
df_update_pivoted = df_update_summed.pivot(index=["step", "block"], columns="surface", values="value").reset_index()  # put sq_sum and num_el side by side
df_update_pivoted["update_ratio"] = np.sqrt(df_update_pivoted["update"] / df_update_pivoted["params"])  # compute update ratio
display(df_update_pivoted.head())
# Plot update ratio by block
plt.figure(figsize=(12, 5))
for block_id, g in df_update_pivoted.groupby("block"):
    plt.plot(g["step"], g["update_ratio"], label=f"layer {block_id}")
plt.title(f"Parameter Update Ratio by Layer (attn+mlp {RUN_SUBTITLE})")
plt.xlabel("step")
plt.ylabel("Update Ratio")
plt.yscale("log")
plt.legend(ncol=2, fontsize=8)
plt.savefig(f"analyze_train_metrics_plot08_param_update_ratio.png", dpi=200, bbox_inches="tight")
plt.show()

## Global Norms

Here only global norms, potential ideas for the future:

- split by AdamW/Muon optimizer
- split by embeddings/transformer matrices/scalars

In [ ]:
# -------------------------------------------------
#     Total Gradient Norm (all parameters)
df_grad_norm_filtered = df[(df['surface'] == 'grad') & (df['stat'] == 'sq_sum')].copy()
df_grad_norm_summed = df_grad_norm_filtered.groupby(['step'], as_index=False)['value'].sum()
df_grad_norm_summed['total_grad_norm'] = np.sqrt(df_grad_norm_summed['value'])
plt.figure(figsize=(12, 5))
plt.plot(df_grad_norm_summed['step'], df_grad_norm_summed['total_grad_norm'])
plt.title(f"Total Gradient Norm (all parameters) ({RUN_SUBTITLE})")
plt.xlabel("step")
plt.ylabel("Total Grad Norm")
plt.yscale("log")
plt.savefig(f"analyze_train_metrics_plot09_total_grad_norm.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# -------------------------------------------------
#     Total Parameter Norm (all parameters)
df_param_norm_filtered = df[(df['surface'] == 'params') & (df['stat'] == 'sq_sum')].copy()
df_param_norm_summed = df_param_norm_filtered.groupby(['step'], as_index=False)['value'].sum()
df_param_norm_summed['total_param_norm'] = np.sqrt(df_param_norm_summed['value'])
plt.figure(figsize=(12, 5))
plt.plot(df_param_norm_summed['step'], df_param_norm_summed['total_param_norm'])
plt.title(f"Total Parameter Norm (all parameters) ({RUN_SUBTITLE})")
plt.xlabel("step")
plt.ylabel("Total Parameter Norm")
plt.yscale("log")
plt.savefig(f"analyze_train_metrics_plot10_total_param_norm.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# -------------------------------------------------
#     Total Update Norm and Ratio (all parameters)
df_update_norm_filtered = df[(df['surface'].isin(['update', 'params'])) & (df['stat'] == 'sq_sum')].copy()
df_update_norm_summed = df_update_norm_filtered.groupby(['step', 'surface'], as_index=False)['value'].sum()
df_update_norm_pivoted = df_update_norm_summed.pivot(index='step', columns='surface', values='value').reset_index()  # put update and params side by side
df_update_norm_pivoted['total_update_norm'] = np.sqrt(df_update_norm_pivoted['update'])
df_update_norm_pivoted['total_update_ratio'] = np.sqrt(df_update_norm_pivoted['update'] / df_update_norm_pivoted['params'])
display(df_update_norm_pivoted.head())
# Plot total update norm
plt.figure(figsize=(12, 5))
plt.plot(df_update_norm_pivoted['step'], df_update_norm_pivoted['total_update_norm'])
plt.title(f"Total Update Norm (all parameters) ({RUN_SUBTITLE})")
plt.xlabel("step")
plt.ylabel("Total Update Norm")
plt.yscale("log")
plt.savefig(f"analyze_train_metrics_plot11_total_update_norm.png", dpi=200, bbox_inches="tight")
plt.show()
# Plot total update ratio
plt.figure(figsize=(12, 5))
plt.plot(df_update_norm_pivoted['step'], df_update_norm_pivoted['total_update_ratio'])
plt.title(f"Total Update Ratio (all parameters) ({RUN_SUBTITLE})")
plt.xlabel("step")
plt.ylabel("Total Update Ratio")
plt.yscale("log")
plt.savefig(f"analyze_train_metrics_plot12_total_update_ratio.png", dpi=200, bbox_inches="tight")
plt.show()

## Logits, Confidence, Entropy

In [ ]:
# -----------------------------------------
#    Logits RMS (reduced across ranks)
df_logits_filtered = df[(df["tensor_name"] == "logits") & (df["surface"] == "fwd")].copy()
df_logits_summed = df_logits_filtered.groupby(["step", "stat"], as_index=False)["value"].sum()  # sum over ranks
df_logits_pivoted = df_logits_summed.pivot(index="step", columns="stat", values="value").reset_index()  # put sq_sum and num_el side by side
df_logits_pivoted['rms'] = np.sqrt(df_logits_pivoted['sq_sum'] / df_logits_pivoted['num_el'])
# display(df_logits_pivoted.head())
plt.figure(figsize=(12, 5))
plt.plot(df_logits_pivoted['step'], df_logits_pivoted['rms'])
plt.title(f"Logits RMS ({RUN_SUBTITLE})")
plt.xlabel("step")
plt.ylabel("Logits RMS")
plt.yscale("log")
plt.savefig(f"analyze_train_metrics_plot13_logits_rms.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------------------
#        Confidence (max prob mean)
df_conf_filtered = df[(df["tensor_name"] == "probs_max") & (df["surface"] == "fwd")].copy()
df_conf_summed = df_conf_filtered.groupby(["step", "stat"], as_index=False)["value"].sum()  # sum over ranks
df_conf_pivoted = df_conf_summed.pivot(index="step", columns="stat", values="value").reset_index()  # put sq_sum and num_el side by side
df_conf_pivoted['max_prob_mean'] = df_conf_pivoted['sum'] / df_conf_pivoted['count']
# display(df_conf_pivoted.head())
plt.figure(figsize=(12, 5))
plt.plot(df_conf_pivoted['step'], df_conf_pivoted['max_prob_mean'])
plt.title(f"Confidence (max prob mean) ({RUN_SUBTITLE})")
plt.xlabel("step")
plt.ylabel("Max Prob Mean")
plt.savefig(f"analyze_train_metrics_plot14_confidence_max_prob_mean.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------
#        Entropy (mean)
df_entropy_filtered = df[(df["tensor_name"] == "entropy") & (df["surface"] == "fwd")].copy()
df_entropy_summed = df_entropy_filtered.groupby(["step", "stat"], as_index=False)["value"].sum()  # sum over ranks
df_entropy_pivoted = df_entropy_summed.pivot(index="step", columns="stat", values="value").reset_index()  # put sq_sum and num_el side by side
df_entropy_pivoted['entropy_mean'] = df_entropy_pivoted['sum'] / df_entropy_pivoted['count']
# display(df_entropy_pivoted.head())
plt.figure(figsize=(12, 5))
plt.plot(df_entropy_pivoted['step'], df_entropy_pivoted['entropy_mean'])
plt.title(f"Entropy (mean) ({RUN_SUBTITLE})")
plt.xlabel("step")
plt.ylabel("Entropy Mean")
plt.savefig(f"analyze_train_metrics_plot15_entropy_mean.png", dpi=200, bbox_inches="tight")
plt.show()

## Select Non-Block Params

In [ ]:
tensor_name = "wte.weight"
#tensor_name = "lm_head.weight"
#tensor_name = "smear_gate.weight"
tensor_name = "value_embed.weight"  # idea: plot by layer

df_tensor_filtered = df[(df["tensor_name"] == tensor_name) & (df["surface"].isin(['grad', 'update', 'params']))].copy()
df_tensor_summed = df_tensor_filtered.groupby(["step", "surface", "stat"], as_index=False)["value"].sum()  # sum over ranks
df_tensor_pivoted = df_tensor_summed.pivot(index="step", columns=["surface", "stat"], values="value").reset_index()  # put sq_sum and num_el side by side
df_tensor_pivoted.columns = ["_".join(str(x) for x in col if x != "") if isinstance(col, tuple) else col for col in df_tensor_pivoted.columns]  # flatten column names
df_tensor_pivoted["grad_rms"] = np.sqrt(df_tensor_pivoted["grad_sq_sum"] / df_tensor_pivoted["grad_num_el"])
df_tensor_pivoted["update_ratio"] = np.sqrt(df_tensor_pivoted["update_sq_sum"] / df_tensor_pivoted["params_sq_sum"])
df_tensor_pivoted["params_norm"] = np.sqrt(df_tensor_pivoted["params_sq_sum"])

plt.figure(figsize=(12, 5))
plt.plot(df_tensor_pivoted["step"], df_tensor_pivoted["grad_rms"])
plt.title(f"{tensor_name} Gradient RMS ({RUN_SUBTITLE})")
plt.xlabel("step")
plt.ylabel(f"{tensor_name} Grad RMS")
plt.yscale("log")
plt.savefig(f"analyze_train_metrics_plot16_{tensor_name}_grad_rms.png", dpi=200, bbox_inches="tight")
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(df_tensor_pivoted["step"], df_tensor_pivoted["update_ratio"])
plt.title(f"{tensor_name} Update Ratio ({RUN_SUBTITLE})")
plt.xlabel("step")
plt.ylabel(f"{tensor_name} Update Ratio")
plt.yscale("log")
plt.savefig(f"analyze_train_metrics_plot17_{tensor_name}_update_ratio.png", dpi=200, bbox_inches="tight")
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(df_tensor_pivoted["step"], df_tensor_pivoted["params_norm"])
plt.title(f"{tensor_name} Parameter Norm ({RUN_SUBTITLE})")
plt.xlabel("step")
plt.ylabel(f"{tensor_name} Params Norm")
plt.yscale("log")
plt.savefig(f"analyze_train_metrics_plot18_{tensor_name}_params_norm.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
df.tensor_name.unique()

In [ ]:
tensor_name = 'resid_lambdas'
#tensor_name = 'x0_lambdas'
#tensor_name = 'smear_lambda'
#tensor_name = 'backout_lambda'

df_scalar_filtered = df[(df["tensor_name"] == tensor_name)].copy()
#assert np.all(df_scalar_filtered[df_scalar_filtered['rank']==0]['value'].values == df_scalar_filtered[df_scalar_filtered['rank']==1]['value'].values)  # ranks should be identical
df_scalar_rank0 = df_scalar_filtered[df_scalar_filtered['rank']==0].copy()  # keep only one rank since they should be identical

plt.figure(figsize=(12, 5))
for idx, df_vals in df_scalar_rank0.groupby("block"):
    plt.plot(df_vals["step"], df_vals["value"], label=f"block {idx}")
plt.title(f"{tensor_name} Value ({RUN_SUBTITLE})")
plt.xlabel("step")
plt.ylabel(f"{tensor_name} Value")
plt.legend()
plt.savefig(f"analyze_train_metrics_plot19_{tensor_name}_value.png", dpi=200, bbox_inches="tight")
plt.show()

# Collage Cells

In [ ]:
RUN_SUBTITLE_2 = "" # f"flops={run_config['target_flops']:.0e}"
# ---------------------------------------------
#      Post-Block Residual RMS by Layer
# Prepare forward residuals for plotting
df_fwd_filtered = df[(df["tensor_name"] == "resid_post") & (df["surface"] == "fwd")].copy()  # filter fwd activations only
df_fwd_summed = df_fwd_filtered.groupby(["step", "block", "stat"], as_index=False)["value"].sum()  # sum over ranks
df_fwd_pivoted = df_fwd_summed.pivot(index=["step", "block"], columns="stat", values="value").reset_index()  # put sq_sum and num_el side by side
df_fwd_pivoted["rms"] = np.sqrt(df_fwd_pivoted["sq_sum"] / df_fwd_pivoted["num_el"])  # compute RMS
display(df_fwd_pivoted.head())
# Plot forward residual RMS by block
(fig, axes) = plt.subplots(1, 3, figsize=(20, 5))
for block_id, g in df_fwd_pivoted.groupby("block"):
    axes[0].plot(g["step"], g["rms"], label=f"layer {block_id}")
axes[0].set_title(f"Post-Block Residual RMS by Layer")
axes[0].set_xlabel("step")
axes[0].set_ylabel("RMS")
axes[0].legend(ncol=2, fontsize=8)

# --------------------------------------------
#    Param Gradient RMS by Layer (attn+mlp)
# Exclude "value_embed.weight", "attn.ve_gate.weight", not part of core block
core_block_tensors = ["attn.c_q.weight", "attn.c_k.weight", "attn.c_v.weight", "attn.c_proj.weight", "mlp.c_fc.weight", "mlp.c_proj.weight"]
# Prepare gradient norms for plotting
df_grad_filtered = df[(df['surface'] == 'grad') & (df["tensor_name"].isin(core_block_tensors))].copy()
df_grad_summed = df_grad_filtered.groupby(["step", "block", "stat"], as_index=False)["value"].sum()  # sum over ranks
df_grad_pivoted = df_grad_summed.pivot(index=["step", "block"], columns="stat", values="value").reset_index()  # put sq_sum and num_el side by side
df_grad_pivoted["rms"] = np.sqrt(df_grad_pivoted["sq_sum"] / df_grad_pivoted["num_el"])  # compute RMS
display(df_grad_pivoted.head())
# Plot gradient RMS by layer
for block_id, g in df_grad_pivoted.groupby("block"):
    axes[1].plot(g["step"], g["rms"], label=f"layer {block_id}")
axes[1].set_title(f"Param Gradient RMS by Layer (attn+mlp{RUN_SUBTITLE_2})")
axes[1].set_xlabel("step")
axes[1].set_ylabel("RMS")
axes[1].set_yscale("log")
axes[1].legend(ncol=2, fontsize=8)

# -------------------------------------------------
#     Parameter Update Ratio by Layer (attn+mlp)
# Exclude "value_embed.weight", "attn.ve_gate.weight", not part of core block
core_block_tensors = ["attn.c_q.weight", "attn.c_k.weight", "attn.c_v.weight", "attn.c_proj.weight", "mlp.c_fc.weight", "mlp.c_proj.weight"]
# Prepare update norms for plotting
df_update_filtered = df[(df['surface'].isin(['update', 'params'])) & (df['stat'] == 'sq_sum') & (df["tensor_name"].isin(core_block_tensors))].copy()
df_update_summed = df_update_filtered.groupby(["step", "block", "surface"], as_index=False)["value"].sum()  # sum over ranks
df_update_pivoted = df_update_summed.pivot(index=["step", "block"], columns="surface", values="value").reset_index()  # put sq_sum and num_el side by side
df_update_pivoted["update_ratio"] = np.sqrt(df_update_pivoted["update"] / df_update_pivoted["params"])  # compute update ratio
display(df_update_pivoted.head())
# Plot update ratio by block
for block_id, g in df_update_pivoted.groupby("block"):
    axes[2].plot(g["step"], g["update_ratio"], label=f"layer {block_id}")
axes[2].set_title(f"Parameter Update Ratio by Layer (attn+mlp{RUN_SUBTITLE_2})")
axes[2].set_xlabel("step")
axes[2].set_ylabel("Update Ratio")
axes[2].set_yscale("log")
axes[2].legend(ncol=2, fontsize=8)

plt.savefig(f"train_metrics_collage.png", dpi=100, bbox_inches="tight")
plt.show()